# Limpieza de Datos — Precios de Propiedades en CABA

A partir del dataset crudo de ZonaProp, limpiamos y transformamos
cada columna para dejarla lista para el análisis.

## Pasos
1. Convertir strings a números
2. Manejar nulos
3. Limpiar barrios
4. Detectar y eliminar outliers
5. Exportar dataset limpio

### 1. Imports y carga

In [34]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("../data/raw/zonaprop_raw.csv")
print(f"Dataset crudo: {df_raw.shape}")
df = df_raw.copy()
df.head()

Dataset crudo: (38298, 7)


,precio,expensas,metros,ambientes,banos,direccion,barrio
0,USD 530.000,$ 770.000 Expensas,172 m² tot.,4 amb.,3 baños,Soldado De La Independencia 1200,"Las Cañitas, Palermo"
1,USD 170.000,NaN,73 m² tot.,4 amb.,1 baño,Malabia al 200,"Villa Crespo, Capital Federal"
2,USD 120.000,$ 260.000 Expensas,54 m² tot.,3 amb.,1 baño,Paraguay e/ Jean Jaures y Doctor Tomás Manuel ...,"Recoleta, Capital Federal"
3,USD 220.000,$ 380.073 Expensas,68 m² tot.,3 amb.,2 baños,PICO 1600. Entre Libertador y 11 De Septiembre,"Núñez, Capital Federal"
4,USD 2.250.000,$ 2.100.000 Expensas,399 m² tot.,4 amb.,3 baños,Av. Alvear 1500,"Recoleta, Capital Federal"


### 2. Limpieza

#### 2.1 Limpieza columna precios

Primero veo si todos los precios estan en el mismo formato "USD X.XXX"

In [35]:
raros = df["precio"][~df["precio"].str.startswith("USD")].value_counts()
print(f"Valores fuera del formato USD: {len(raros)} tipos distintos")
print(raros)

Valores fuera del formato USD: 4 tipos distintos
precio
Consultar precio    175
$ 265.000             1
$ 110.000             1
$ 142.000             1
Name: count, dtype: int64


Hay 175 propiedades sin precio y otras 3 con el precio es Pesos Argentinos.
Son solo 178 filas sobre 35k, menos del 0.5%, las elimino ya que no se pierde nada relevante

In [36]:
def limpiar_precio(valor):
    if pd.isna(valor):
        return None
    # Eliminar casos sin precio numérico en USD
    if not valor.startswith("USD"):
        return None
    try:
        valor = valor.replace("USD ", "").replace(".", "")
        return int(valor)
    except ValueError:
        return None

df["precio"] = df["precio"].apply(limpiar_precio)

# Eliminar filas sin precio
df = df.dropna(subset=["precio"])

print(f"Filas después de limpiar precio: {len(df)}")
print(f"Filas eliminadas: {len(df_raw) - len(df)}")
print(f"\nMin: USD {df['precio'].min():,}")
print(f"Max: USD {df['precio'].max():,}")
print(f"Media: USD {df['precio'].mean():,.0f}")

Filas después de limpiar precio: 38120
Filas eliminadas: 178

Min: USD 1.0
Max: USD 50,520,900.0
Media: USD 400,594


#### 2.2 Limpieza columna expensas

Primero veo si todas las exprensas estan en el mismo formato "$ XXX.XXX Expensas"

In [37]:
raros_exp = df["expensas"][
    df["expensas"].notna() & 
    ~df["expensas"].str.startswith("$")
].value_counts()

print(f"Valores fuera del formato: {len(raros_exp)} tipos distintos")
print(raros_exp)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


Todas las expensas estan en el mismo formato

In [38]:
def limpiar_expensas(valor):
    if pd.isna(valor):
        return 0  # sin expensas = 0
    try:
        valor = valor.replace("$ ", "").replace(" Expensas", "").replace(".", "")
        return int(valor)
    except ValueError:
        return 0

df["expensas"] = df["expensas"].apply(limpiar_expensas)

print(f"Nulos: {df['expensas'].isnull().sum()}")
print(f"Ceros (sin expensas): {(df['expensas'] == 0).sum()}")
print(f"\nMin: $ {df['expensas'].min():,}")
print(f"Max: $ {df['expensas'].max():,}")
print(f"Media: $ {df['expensas'].mean():,.0f}")

Nulos: 0
Ceros (sin expensas): 15869

Min: $ 0
Max: $ 48,090,200
Media: $ 226,044


#### 2.3 Limpieza metros colummna

Veo si todos los valores en la columna siguen el mismo formato "Xm² tot."

In [39]:
raros_metros = df["metros"][
    df["metros"].notna() &
    ~df["metros"].str.contains("m²")
].value_counts()

print(f"Valores fuera del formato: {len(raros_metros)} tipos distintos")
print(raros_metros)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


Hago el formateo y ademas elimino los valores nulos (18 en total).

In [40]:
def limpiar_metros(valor):
    if pd.isna(valor):
        return None
    try:
        valor = valor.replace(" m² tot.", "").replace(".", "")
        return int(valor)
    except ValueError:
        return None

df["metros"] = df["metros"].apply(limpiar_metros)

# Eliminar los 18 nulos
df = df.dropna(subset=["metros"])

print(f"Filas después de limpiar metros: {len(df)}")
print(f"\nMin: {df['metros'].min()} m²")
print(f"Max: {df['metros'].max()} m²")
print(f"Media: {df['metros'].mean():.0f} m²")

Filas después de limpiar metros: 38102

Min: 1.0 m²
Max: 3500000.0 m²
Media: 473 m²


#### 2.4 Limpieza columna ambientes

Veo que todas las entradas tengan el mismo formato "X amb."

In [41]:
raros_amb = df["ambientes"][
    df["ambientes"].notna() &
    ~df["ambientes"].str.contains("amb")
].value_counts()

print(f"Valores fuera del formato: {len(raros_amb)} tipos distintos")
print(raros_amb)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


Formateo:

In [42]:
def limpiar_ambientes(valor):
    if pd.isna(valor):
        return None
    try:
        valor = valor.replace(" amb.", "")
        return int(valor)
    except ValueError:
        return None

df["ambientes"] = df["ambientes"].apply(limpiar_ambientes)

print(f"Nulos: {df['ambientes'].isnull().sum()}")
print(f"\nMin: {df['ambientes'].min()}")
print(f"Max: {df['ambientes'].max()}")
print(f"Media: {df['ambientes'].mean():.1f}")

Nulos: 3217

Min: 1.0
Max: 56.0
Media: 3.1


### 2.5 Limpieza columna banos

In [43]:
raros_banos = df["banos"][
    df["banos"].notna() &
    ~df["banos"].str.contains("baño")
].value_counts()

print(f"Valores fuera del formato: {len(raros_banos)} tipos distintos")
print(raros_banos)

Valores fuera del formato: 0 tipos distintos
Series([], Name: count, dtype: int64)


In [44]:
def limpiar_banos(valor):
    if pd.isna(valor):
        return None
    try:
        valor = valor.replace(" baños", "").replace(" baño", "")
        return int(valor)
    except ValueError:
        return None

df["banos"] = df["banos"].apply(limpiar_banos)

print(f"Nulos: {df['banos'].isnull().sum()}")
print(f"\nMin: {df['banos'].min()}")
print(f"Max: {df['banos'].max()}")
print(f"Media: {df['banos'].mean():.1f}")

Nulos: 2625

Min: 1.0
Max: 70.0
Media: 1.7
